# 🔬 Walking skeleton v0 — open the hood

A hands-on tour of the M0.3 walking skeleton. We build Ada's purchase of **50 Mbps**
from Bell (ticket **#7**, **10 TOK**) **one component at a time**, and *inspect every
object* as it appears — nothing hidden behind a wrapper.

> **How to run:** in VS Code, click **▶▶ Run All**, or step through with **Shift+Enter**.
> Each code cell builds or pokes at a piece; the markdown above it says what to look for.
> A *cell* is just a chunk of code that runs on its own and shows its last value.
>
> Narrative companion: [`docs/03c-skeleton-walkthrough.md`](../../docs/03c-skeleton-walkthrough.md).
> Everything here calls the same code the lifecycle tests assert (`e2e/tests/test_lifecycle.py`).

## 0. The cast — what we import

Three groups: the **shapes** (validated data types from `a2a_interfaces`, M0.2), the
**fixtures** (`fx` — the one canonical Ada/Bell/#7 example), and the **skeleton** (the
cardboard fakes + the stub controller, M0.3).

In [1]:
# shapes: the two "ports" (interfaces) our fakes must satisfy
from a2a_interfaces import EntitlementReader, NetworkProvisioner

# the canonical example, in ONE place so story/docs/tests never disagree
from a2a_interfaces import fixtures as fx

# the walking skeleton: cardboard props + the stub controller
from e2e.skeleton.fakes import FakeChain, FakeClock, FakeNet, OfferAlreadyUsed
from e2e.skeleton.scripted_agents import ScriptedConsumer, ScriptedProvider
from e2e.skeleton.stub_controller import StubController, predicate

print("imported: 2 ports, the fixtures (fx), and the skeleton")

imported: 2 ports, the fixtures (fx), and the skeleton


## 1. What is Ada even buying? Inspect the data first

Before any action, look at the *shapes*. These are **pydantic models** — Python objects
that validate their fields on creation. `.model_dump()` turns one into a plain dict so you
can read every field.

In [2]:
need = fx.BANDWIDTH_NEED          # what Ada wants
print("type:", type(need).__name__)
need.model_dump()                 # last expression -> the notebook displays it

type: BandwidthNeed


{'v': 0,
 'kind': 'bandwidth',
 'src': 'hostA',
 'dst': 'hostB',
 'capacity_bps': 50000000,
 'qos_class': 1,
 'window': {'start': 1757944800, 'end': 1757952000}}

`src`/`dst` are **catalog** names (`hostA`→`hostB`) from Bell's menu — *not* device
names. The paper-to-physics mapping is the controller's secret (ADR-005). Now the offer
Bell signs — twelve fields that mirror the on-chain struct exactly:

In [3]:
offer = fx.CANONICAL_OFFER
print("Offer has", len(offer.model_dump()), "fields (mirrors the Solidity EIP-712 struct)")
offer.model_dump()

Offer has 12 fields (mirrors the Solidity EIP-712 struct)


{'provider': '0x70997970C51812dc3A010C7d01b50e0d17dc79C8',
 'consumer': '0x0000000000000000000000000000000000000000',
 'service_type': 0,
 'resource_id': '0x0000000000000000000000000000000000000000000000000000000000000007',
 'params': '0x0000000000000000000000000000000000000000000000000000000002faf0800000000000000000000000000000000000000000000000000000000000000001',
 'start_time': 1757944800,
 'end_time': 1757952000,
 'payment_token': '0x5FbDB2315678afecb367f032d93F642f64180aa3',
 'price': '10000000000000000000',
 'valid_until': 1757946000,
 'salt': '0x0000000000000000000000000000000000000000000000000000000000005a17',
 'terms_hash': '0x2222222222222222222222222222222222222222222222222222222222222222'}

In [4]:
# An Offer wrapped with its signature + the human-readable SLA = a SignedOffer
signed = fx.CANONICAL_SIGNED_OFFER
print("provider (Bell):", signed.offer.provider)
print("price          :", int(signed.offer.price) // 10**18, "TOK")
print("signature      :", signed.signature[:14], "...  <- a PLACEHOLDER; fakes don't verify it")
signed.terms_doc

provider (Bell): 0x70997970C51812dc3A010C7d01b50e0d17dc79C8
price          : 10 TOK
signature      : 0xabababababab ...  <- a PLACEHOLDER; fakes don't verify it


{'sla': {'latency_ms': 20, 'loss_pct': 0.1},
 'notes': 'best effort beyond rate'}

The two players are just **addresses** (account ids on the chain):

In [5]:
print("Ada  (buyer)        :", fx.ADA)
print("Bell (provider/issuer):", fx.BELL)
print("ticket id we'll mint  :", fx.TICKET_ID)

Ada  (buyer)        : 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
Bell (provider/issuer): 0x70997970C51812dc3A010C7d01b50e0d17dc79C8
ticket id we'll mint  : 7


## 2. A clock you can fast-forward (`FakeClock`)

Validity is measured by **chain time** (ADR-004). We can't wait two real hours, so the
fake clock is just one integer we shove forward with `advance()`.

In [6]:
clock = FakeClock(fx.WINDOW.start - 1800)     # 30 min before the window opens (~13:30)
print("now (unix secs):", clock.now())
clock.advance(1800)                           # jump forward 30 minutes
print("after advance  :", clock.now(), "== window start?", clock.now() == fx.WINDOW.start)

now (unix secs): 1757943000
after advance  : 1757944800 == window start? True


## 3. The blockchain... is a dict (`FakeChain`)

`FakeChain` keeps only the bookkeeping: two **balances**, a **set** of spent salts, an
**owners** map, an **entitlements** map. Build one and look inside. (Names with a leading
`_` are "internal" — we peek only to learn.)

In [7]:
clock = FakeClock(fx.WINDOW.start - 1800)
chain = FakeChain(clock, balances={fx.ADA: 50 * 10**18, fx.BELL: 0}, next_id=fx.TICKET_ID)
print("balances :", {a[:6] + '...': v // 10**18 for a, v in chain.balances.items()}, "(TOK)")
print("consumed :", chain.consumed, "  <- no offer salts spent yet")
print("owners   :", chain._owners, "  <- no tickets minted yet")

balances : {'0xf39F...': 50, '0x7099...': 0} (TOK)
consumed : set()   <- no offer salts spent yet
owners   : {}   <- no tickets minted yet


`FakeChain` **is** an `EntitlementReader` — not by inheritance, but by *shape*: it has
the methods the port requires. That structural fit is the whole trick — the real
`chainmcp` client drops into the same hole later with no other change.

In [8]:
print("FakeChain satisfies EntitlementReader?", isinstance(chain, EntitlementReader))
print("port methods:", [m for m in dir(EntitlementReader) if not m.startswith('_')])

FakeChain satisfies EntitlementReader? True
port methods: ['chain_time', 'get', 'owner_of', 'watch_revoked']


## 4. The one on-chain moment: `fulfill()`

Bell quotes, Ada accepts (both **faked** decisions — the only LLM slots in the real
system), then Ada *redeems* the offer. Watch balances move and a ticket appear — all or
nothing.

In [9]:
provider = ScriptedProvider()
consumer = ScriptedConsumer()

signed = provider.quote(need)               # Bell's slot (canned SignedOffer)
decision = consumer.decide(need, signed)    # Ada's slot (canned accept)
print("Ada's decision:", decision.model_dump())

Ada's decision: {'accept': True, 'reason': 'meets need; price within budget'}


In [10]:
print("BEFORE fulfill:")
print("  Ada  :", chain.balances[fx.ADA] // 10**18, "TOK")
print("  Bell :", chain.balances[fx.BELL] // 10**18, "TOK")
print("  salts spent:", len(chain.consumed))

BEFORE fulfill:
  Ada  : 50 TOK
  Bell : 0 TOK
  salts spent: 0


In [11]:
eid = chain.fulfill(signed, buyer=fx.ADA)
print("fulfill() returned entitlement id:", eid)

fulfill() returned entitlement id: 7


In [12]:
print("AFTER fulfill:")
print("  Ada  :", chain.balances[fx.ADA] // 10**18, "TOK   (paid 10)")
print("  Bell :", chain.balances[fx.BELL] // 10**18, "TOK   (received 10)")
print("  salts spent:", len(chain.consumed), "->", {s[:10] + '...' for s in chain.consumed})
print("  owner of #" + str(eid) + " is Ada?", chain.owner_of(eid) == fx.ADA)

AFTER fulfill:
  Ada  : 40 TOK   (paid 10)
  Bell : 10 TOK   (received 10)
  salts spent: 1 -> {'0x00000000...'}
  owner of #7 is Ada? True


The mint created an `EntitlementView` — the controller's read-only view of ticket #7.
Its `params` is a decoded `BandwidthParams`:

In [13]:
view = chain.get(eid)
print("type     :", type(view).__name__)
print("id       :", view.id)
print("issuer   :", view.issuer, "(Bell)")
print("capacity :", view.params.capacity_bps // 1_000_000, "Mbps")
print("qos_class:", view.params.qos_class)
print("window   :", view.start_time, "->", view.end_time)
print("revoked  :", view.revoked)

type     : EntitlementView
id       : 7
issuer   : 0x70997970C51812dc3A010C7d01b50e0d17dc79C8 (Bell)
capacity : 50 Mbps
qos_class: 1
window   : 1757944800 -> 1757952000
revoked  : False


**Single-use & atomicity.** Spend the *same* offer twice → it's rejected, and the world
is unchanged (the salt is "punched" once). We catch the error with `try/except`:

In [14]:
try:
    chain.fulfill(signed, buyer=fx.ADA)        # the same offer again
except OfferAlreadyUsed as e:
    print("replay REJECTED — salt already used:", str(e)[:14], "...")

print("Bell still has", chain.balances[fx.BELL] // 10**18, "TOK (not 20 — nothing double-charged)")

replay REJECTED — salt already used: 0x000000000000 ...
Bell still has 10 TOK (not 20 — nothing double-charged)


## 5. The bouncer's checklist: `predicate()` (a pure function)

The authorization decision is **pure** — hand it the facts, it returns `None` (allowed)
or the first failing `ErrorCode`. No chain, no network: you can call it by hand.

In [15]:
now = fx.WINDOW.start + 120        # 14:02, inside the window
verdict = predicate(view, owner=fx.ADA, requester=fx.ADA, now=now, active_ids=set())
print("inside window, correct owner ->", verdict, " (None means ALLOWED)")

inside window, correct owner -> None  (None means ALLOWED)


In [16]:
# flip ONE fact at a time and watch the matching denial:
print("wrong requester ->", predicate(view, fx.ADA, fx.BELL, now, set()))
print("too early       ->", predicate(view, fx.ADA, fx.ADA, fx.WINDOW.start - 1, set()))
print("window over     ->", predicate(view, fx.ADA, fx.ADA, fx.WINDOW.end, set()))
print("already active  ->", predicate(view, fx.ADA, fx.ADA, now, {view.id}))

revoked_view = view.model_copy(update={"revoked": True})   # frozen model -> make a copy
print("revoked ticket  ->", predicate(revoked_view, fx.ADA, fx.ADA, now, set()))

wrong requester -> ErrorCode.E_NOT_OWNER
too early       -> ErrorCode.E_NOT_STARTED
window over     -> ErrorCode.E_EXPIRED
already active  -> ErrorCode.E_CONFLICT
revoked ticket  -> ErrorCode.E_REVOKED


## 6. The router that just takes notes (`FakeNet`)

It *records* `apply_*`/`teardown` calls instead of touching hardware. Build it and confirm
it fits the `NetworkProvisioner` port.

In [17]:
net = FakeNet()
print("satisfies NetworkProvisioner?", isinstance(net, NetworkProvisioner))
print("applied (empty):", net.applied)
print("health():", net.health())

satisfies NetworkProvisioner? True
applied (empty): {}
health(): True


## 7. The bouncer, wired up: `StubController`

It depends only on the two **ports** (chain, net) — that's why fakes work here and real
adapters work later. Activation is **fetch facts → run predicate → act**. First, building
it auto-registers a revoke-watcher on the chain:

In [18]:
controller = StubController(chain, net)
print("controller built. It registered its _on_revoked callback on the chain:")
print("  chain watchers:", len(chain._watchers))

controller built. It registered its _on_revoked callback on the chain:
  chain watchers: 1


**Challenge → proof → activate.** The controller issues a one-time `nonce`; Ada presents
it back (a faked "proof" in v0). The nonce is burned on use, so it can't be replayed.

In [19]:
nonce = controller.challenge(eid)
print("challenge() -> nonce:", nonce)
print("open nonces:", controller._open_nonces)

challenge() -> nonce: nonce-0
open nonces: {'nonce-0'}


In [20]:
clock.advance((fx.WINDOW.start + 120) - clock.now())     # move chain time to 14:02
sid = controller.activate(eid, requester=fx.ADA, nonce=nonce)
print("activate() -> session id:", sid)
print("session state:", controller.state(sid).value)
print("nonce burned? open nonces now:", controller._open_nonces)

activate() -> session id: sess-0
session state: active
nonce burned? open nonces now: set()


The controller translated ticket #7's `resource_id` into a concrete device/path and
called the router. See exactly what `FakeNet` recorded:

In [21]:
cfg = net.applied[sid]
print("device:", cfg["path"].device)
print("path  :", cfg["path"].ingress_if, "->", cfg["path"].egress_if)
print("rate  :", cfg["capacity_bps"] // 1_000_000, "Mbps")
print("qos   :", cfg["qos_class"])

device: srl1
path  : ethernet-1/1 -> ethernet-1/2
rate  : 50 Mbps
qos   : 1


## 8. Teardown path A — time runs out (`tick()`)

Advance the clock past the window end and `tick()`. The controller re-reads chain time
(ADR-004) and tears the session down.

In [22]:
clock.advance(fx.WINDOW.end - clock.now())        # jump to 16:00
print("before tick:", controller.state(sid).value)
controller.tick()
print("after tick :", controller.state(sid).value)
print("net.applied :", net.applied, "  <- config removed")
print("net.torn_down:", net.torn_down)

before tick: active
after tick : torn_down
net.applied : {}   <- config removed
net.torn_down: ['sess-0']


## 9. Teardown path B — Bell revokes (`watch_revoked` callback)

A fresh run: settle, activate, then Bell **revokes** mid-window. The chain fires the
callback the controller registered, so it tears down **immediately — no `tick()` needed**.
This is the event-driven path (the time path above is polling).

In [23]:
clk2 = FakeClock(fx.WINDOW.start + 120)           # start already inside the window
chain2 = FakeChain(clk2, balances={fx.ADA: 50 * 10**18, fx.BELL: 0}, next_id=fx.TICKET_ID)
net2 = FakeNet()
ctrl2 = StubController(chain2, net2)

eid2 = chain2.fulfill(provider.quote(need), buyer=fx.ADA)
sid2 = ctrl2.activate(eid2, requester=fx.ADA, nonce=ctrl2.challenge(eid2))
print("before revoke:", ctrl2.state(sid2).value)

chain2.revoke(eid2)        # Bell cancels -> watch_revoked fires -> controller tears down
print("after revoke :", ctrl2.state(sid2).value, " (event-driven, no tick)")
print("net torn_down:", net2.torn_down)

before revoke: active
after revoke : torn_down  (event-driven, no tick)
net torn_down: ['sess-0']


## What you just did

You built and inspected **every component** of the walking skeleton and ran the whole use
case by hand:

| piece | you saw |
|---|---|
| shapes (`fx.*`) | the validated `Offer` / `EntitlementView` fields |
| `FakeClock` | chain time you can `advance()` |
| `FakeChain` | balances move, salt punched, ticket minted; replay rejected |
| `predicate()` | one accept, five named denials — pure, callable by hand |
| `FakeNet` | the 50 Mbps config it recorded |
| `StubController` | challenge → activate → ACTIVE, then teardown two ways |

The four cardboard props get swapped for real organs later, one at a time:
**FakeChain → chainmcp (M1.5)**, **FakeNet → netctl (M3.4)**,
**StubController → controller (M4.x)**, **scripted agents → LLM agents (M5.6)** — and
because each fits a *port*, the wiring above never changes.

**Check question:** in §7, `controller.activate(...)` called `chain.get(eid)` on a
`FakeChain`. When that becomes the real `chainmcp` at M1.5, why does `activate` need *zero*
changes — and which line in §3 proved the fake and the real one are interchangeable?